# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
#your code here
# Preprocessing pipeline

# Drop columns that are not useful for modeling
spaceship = spaceship.drop(columns=['PassengerId', 'Name', 'Cabin'])

# Handle missing values
# For numerical columns: fill with median
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median())

# For categorical columns: fill with mode
cat_cols = ['HomePlanet', 'Destination', 'CryoSleep', 'VIP']
for col in cat_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].mode()[0])

# Convert boolean columns to int (CryoSleep, VIP, Transported)
spaceship['CryoSleep'] = spaceship['CryoSleep'].astype(int)
spaceship['VIP'] = spaceship['VIP'].astype(int)
spaceship['Transported'] = spaceship['Transported'].astype(int)

# One-hot encode categorical features
spaceship = pd.get_dummies(spaceship, columns=['HomePlanet', 'Destination'], drop_first=True)

# Separate features and target
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported']

# Feature scaling (standardization)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8584\923368845.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spaceship[col] = spaceship[col].fillna(spaceship[col].mode()[0])


**Perform Train Test Split**

In [6]:
#your code here
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [7]:
#your code here
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Bagging 
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, bootstrap=True, random_state=42)
bagging.fit(X_train, y_train)
print(f"Bagging Accuracy: {bagging.score(X_test, y_test):.4f}")

# Pasting 
pasting = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, bootstrap=False, random_state=42)
pasting.fit(X_train, y_train)
print(f"Pasting Accuracy: {pasting.score(X_test, y_test):.4f}")

Bagging Accuracy: 0.7775
Pasting Accuracy: 0.7315


- Random Forests

In [8]:
#your code here
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest Accuracy: {rf.score(X_test, y_test):.4f}")

Random Forest Accuracy: 0.7798


- Gradient Boosting

In [9]:
#your code here
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
print(f"Gradient Boosting Accuracy: {gb.score(X_test, y_test):.4f}")

Gradient Boosting Accuracy: 0.7838


- Adaptive Boosting

In [10]:
#your code here
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
print(f"AdaBoost Accuracy: {ada.score(X_test, y_test):.4f}")

AdaBoost Accuracy: 0.7435


Which model is the best and why?

In [12]:
#comment here
# Compare all accuracies (optional: store them in a dictionary)
results = {
    'Bagging': bagging.score(X_test, y_test),
    'Pasting': pasting.score(X_test, y_test),
    'Random Forest': rf.score(X_test, y_test),
    'Gradient Boosting': gb.score(X_test, y_test),
    'AdaBoost': ada.score(X_test, y_test)
}
best_model = max(results, key=results.get)
print(f"Best model: {best_model} with accuracy {results[best_model]:.4f}")


Best model: Gradient Boosting with accuracy 0.7838
